# Movies Recommender System

In this project, we'll be building a Movie Recommendation System using [TMDB 5000 Movie Dataset](https://www.kaggle.com/tmdb/tmdb-movie-metadata). 

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from ast import literal_eval
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import linear_kernel, cosine_similarity
from nltk.stem.snowball import SnowballStemmer
from nltk.stem.wordnet import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords, wordnet

import string
import re

import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/asitdave/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /Users/asitdave/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/asitdave/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/asitdave/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [2]:
md = pd.read_csv('movies_metadata.csv')
md.head()

/var/folders/xy/2yvnpx8d56vfjg1nvw677xx80000gn/T/ipykernel_57890/2986690614.py:1: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  md = pd.read_csv('movies_metadata.csv')


,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0
3,False,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",...,1995-12-22,81452156.0,127.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Friends are the people who let you be yourself...,Waiting to Exhale,False,6.1,34.0
4,False,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,...,1995-02-10,76578911.0,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173.0


In [3]:
# Check for missing values
md.isnull().sum()

adult                        0
belongs_to_collection    40972
budget                       0
genres                       0
homepage                 37684
id                           0
imdb_id                     17
original_language           11
original_title               0
overview                   954
popularity                   5
poster_path                386
production_companies         3
production_countries         3
release_date                87
revenue                      6
runtime                    263
spoken_languages             6
status                      87
tagline                  25054
title                        6
video                        6
vote_average                 6
vote_count                   6
dtype: int64

In [4]:
def clean_genres(row):
    genres = [item['name'] for item in literal_eval(row['genres'])]
    return genres

md['genres'] = md.apply(clean_genres, axis=1)

In [5]:
md['genres']

0         [Animation, Comedy, Family]
1        [Adventure, Fantasy, Family]
2                   [Romance, Comedy]
3            [Comedy, Drama, Romance]
4                            [Comedy]
                     ...             
45461                 [Drama, Family]
45462                         [Drama]
45463       [Action, Drama, Thriller]
45464                              []
45465                              []
Name: genres, Length: 45466, dtype: object

In [6]:
md['release_date'] = pd.to_datetime(md['release_date'], errors='coerce')
md['year'] = md['release_date'].dt.year


In [7]:
md['year']

0        1995.0
1        1995.0
2        1995.0
3        1995.0
4        1995.0
          ...  
45461       NaN
45462    2011.0
45463    2003.0
45464    1917.0
45465    2017.0
Name: year, Length: 45466, dtype: float64

We use IMDB's *weighted rating* formula to construct the chart. Mathematically, it is represented as follows:

Weighted Rating (WR) = $(\frac{v}{v + m} \cdot R) + (\frac{m}{v + m} \cdot C)$

where,
* *v* is the number of votes for the movie
* *m* is the minimum votes required to be listed in the chart
* *R* is the average rating of the movie
* *C* is the mean vote across the whole report

The next step is to determine an appropriate value for *m*, the minimum votes required to be listed in the chart. We will use **95th percentile** as our cutoff. In other words, for a movie to feature in the charts, it must have more votes than at least 95% of the movies in the list.

We build our overall Top 250 Chart and define a function to build charts for a particular genre.

In [8]:
# Calculate C, the mean of the movie ratings
C = md[md['vote_average'].notnull()]['vote_average'].mean()
C

5.618207215134184

In [9]:
m = md['vote_count'].quantile(0.95)
m

434.0

In [10]:
def weighted_rating(row, m=m, C=C):
    v = row['vote_count']
    R = row['vote_average']
    
    # Calculation based on the IMDB formula
    return (v / (v + m) * R) + (m / (m + v) * C)

In [11]:
# Find all qualified movies
qualified_movies = md.copy().loc[(md['vote_count'] >= m) & (md['vote_count'].notnull()) & (md['vote_average'].notnull())]
qualified_movies['vote_count'] = qualified_movies['vote_count'].astype('int')
qualified_movies['vote_average'] = qualified_movies['vote_average'].astype('int')
qualified_movies.shape

(2274, 25)

Total of **2274 movies** qualify to be considered for the chart, given the condition: A movie has to have at least 434 votes on TMDB.

In [12]:
qualified_movies['ratings'] = qualified_movies.apply(weighted_rating, axis=1)

In [13]:
qualified_movies = qualified_movies[['title', 'year', 'ratings', 'popularity', 'genres']]
qualified_movies['year'] = qualified_movies['year'].astype('int')

#### Top 10 movies

In [14]:
qualified_movies.sort_values(by='ratings', ascending=False).head(10)

,title,year,ratings,popularity,genres
15480,Inception,2010,7.928755,29.108149,"[Action, Thriller, Science Fiction, Mystery, A..."
12481,The Dark Knight,2008,7.918626,123.167259,"[Drama, Action, Crime, Thriller]"
22879,Interstellar,2014,7.911049,32.213481,"[Adventure, Drama, Science Fiction]"
2843,Fight Club,1999,7.897775,63.869599,[Drama]
4863,The Lord of the Rings: The Fellowship of the Ring,2001,7.889160,32.070725,"[Adventure, Fantasy, Action]"
292,Pulp Fiction,1994,7.886457,140.950236,"[Thriller, Crime]"
314,The Shawshank Redemption,1994,7.882427,51.645403,"[Drama, Crime]"
7000,The Lord of the Rings: The Return of the King,2003,7.880635,29.324358,"[Adventure, Fantasy, Action]"
351,Forrest Gump,1994,7.879536,48.307194,"[Comedy, Drama, Romance]"
5814,The Lord of the Rings: The Two Towers,2002,7.871988,29.423537,"[Adventure, Fantasy, Action]"


## Create chart based on genre(s)

In [15]:
def build_chart(genre: list[str], percentile=0.95):
    # Filter movies based on genre(s)
    df = md[md['genres'].apply(lambda x: all(g in x for g in genre))]

    vote_counts = df[df['vote_count'].notnull()]['vote_count'].astype('int')
    vote_averages = df[df['vote_average'].notnull()]['vote_average'].astype('int')
    C = vote_averages.mean()
    m = vote_counts.quantile(percentile)

    qualified = df.copy()[(df['vote_count'] >= m) & (df['vote_count'].notnull()) & (df['vote_average'].notnull())]
    qualified['vote_count'] = qualified['vote_count'].astype('int')
    qualified['vote_average'] = qualified['vote_average'].astype('int')
    qualified['year'] = qualified['year'].astype('int')
    
    qualified['ratings'] = qualified.apply(weighted_rating, axis=1)
    qualified = qualified.sort_values('ratings', ascending=False).head(250)
    df = qualified[['title', 'year', 'ratings', 'popularity', 'vote_count', 'vote_average']]
    return df#.set_index('title')

In [16]:
build_chart(['Action', 'Romance']).head(10)

,title,year,ratings,popularity,vote_count,vote_average
23437,Maleficent,2014,6.881036,19.467404,4607,7
3871,"Crouching Tiger, Hidden Dragon",2000,6.566379,19.944953,949,7
6212,Castle in the Sky,1986,6.542564,11.061551,877,7
550,True Romance,1993,6.498580,17.189328,762,7
1357,The Last of the Mohicans,1992,6.492212,15.228794,747,7
8256,House of Flying Daggers,2004,6.323140,9.3002,452,7
15265,Prince of Persia: The Sands of Time,2010,5.940949,18.459955,2372,6
20670,The Host,2013,5.927737,11.342024,1859,6
16220,The Tourist,2010,5.923922,12.845045,1744,6
1069,Top Gun,1986,5.923641,20.301019,1736,6


In [17]:
build_chart(['Romance']).head(10)

,title,year,ratings,popularity,vote_count,vote_average
351,Forrest Gump,1994,7.879536,48.307194,8147,8
10309,Dilwale Dulhania Le Jayenge,1995,7.659636,34.457024,661,9
876,Vertigo,1958,7.352320,18.20822,1162,8
40251,Your Name.,2016,7.293922,34.461252,1030,8
883,Some Like It Hot,1959,7.185423,11.845107,835,8
1132,Cinema Paradiso,1988,7.184781,14.177005,834,8
19901,Paperman,2012,7.114985,7.198633,734,8
37863,Sing Street,2016,7.062830,10.672862,669,8
1639,Titanic,1997,6.926902,26.88907,7770,7
882,The Apartment,1960,6.890882,11.994281,498,8


## Content Based Recommender

The recommender we built in the previous section suffers some severe limitations. For example, it gives the same recommendation to everyone, regardless of the user's personal taste or history.

To personalise our recommendations more, we build an engine that computes similarity between movies based on certain metrics and suggests movies that are most similar to a particular movie that a user liked. Since we will be using movie metadata (or content) to build this engine, this also known as **Content Based Filtering.**

We will build two Content Based Recommenders based on:
* Movie Overviews and Taglines
* Movie Cast, Crew, Keywords and Genre

##### 1. Recommendation using Movie Overviews and Taglines

In [18]:
md['overview']

0        Led by Woody, Andy's toys live happily in his ...
1        When siblings Judy and Peter discover an encha...
2        A family wedding reignites the ancient feud be...
3        Cheated on, mistreated and stepped on, the wom...
4        Just when George Banks has recovered from his ...
                               ...                        
45461          Rising and falling between a man and woman.
45462    An artist struggles to finish his work while a...
45463    When one of her hits goes wrong, a professiona...
45464    In a small town live two brothers, one a minis...
45465    50 years after decriminalisation of homosexual...
Name: overview, Length: 45466, dtype: object

In [19]:
md['title'].duplicated().sum()

3188

In [20]:
md['budget'] = pd.to_numeric(md['budget'], errors='coerce').astype('Int64')

In [21]:
# Remove movie duplicates using budget column
md = md.sort_values(by='budget', ascending=False)
md = md.drop_duplicates(subset='title', keep='first')

In [22]:
md.shape

(42278, 25)

#### Preprocess the text

In [23]:
def filter_numbers(text):
    # # Remove all numbers from the text
    # return re.sub(r'\d+', '', text)
    #Alternatively, replace numbers with a placeholder
    return re.sub(r'\d+', ' <NUM> ', text)

# Define a TF-IDF Vectorizer Object and remove all english stop words
tfidf = TfidfVectorizer(min_df=2, max_df=0.85,
                        lowercase=True, 
                        stop_words='english',
                        analyzer='word',
                        preprocessor=filter_numbers,
                        # ngram_range=(1, 2)
                        )

# Combine movie descriptions and taglines
md['tagline'] = md['tagline'].fillna('')
md['tag_and_description'] = md['overview'] + md['tagline']# + md['genres'].apply(lambda x: ' '.join(x) if isinstance(x, list) else '')
# md['tag_and_description'] = md['genres'].apply(lambda x: ' '.join(x) if isinstance(x, list) else '')
md['tag_and_description'] = md['tag_and_description'].fillna('')

# Construct the required TF-IDF matrix by fitting and transforming the data
tfidf_matrix = tfidf.fit_transform(md['tag_and_description'])

# Output the shape of tfidf_matrix
tfidf_matrix.shape

(42278, 43281)

In [24]:
# Compute the cosine similarity matrix
cosine_sim_arr = linear_kernel(tfidf_matrix, tfidf_matrix)
cosine_sim_arr.shape

(42278, 42278)

In [25]:
md2 = md.copy().reset_index()
indices = pd.Series(md2.index, index=md2['title'])

In [26]:
def recommend_movie(title, top_n=10, cosine_sim_arr=cosine_sim_arr):

    def check_movies(movie):
        if movie in md['title'].values:
            return True
        else:
            return False

    if not check_movies(title):
        return "Movie not found. Please check the title."
    
    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim_arr[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n+1]  # skip the movie itself
    movie_indices = [i[0] for i in sim_scores]

    return md['title'].iloc[movie_indices]

In [27]:
recommend_movie("The Godfather")

1178               The Godfather: Part II
44030    The Godfather Trilogy: 1972-1990
1914              The Godfather: Part III
23126                          Blood Ties
31973                    Honor Thy Father
11297                    Household Saints
29567                          Blood Feud
30197                    See You Tomorrow
30358             Quelle strane occasioni
31537                         Sorelle Mai
Name: title, dtype: object

In [28]:
recommend_movie("The Dark Knight")

18252                                The Dark Knight Rises
1328                                        Batman Returns
21194    Batman Unmasked: The Psychology of the Dark Kn...
150                                         Batman Forever
41976                                The Lego Batman Movie
15511                           Batman: Under the Red Hood
20232              Batman: The Dark Knight Returns, Part 2
25267                                    Batman vs Dracula
40974    LEGO DC Comics Super Heroes: Batman: Be-Leaguered
41982    Batman Beyond Darwyn Cooke's Batman 75th Anniv...
Name: title, dtype: object

In [29]:
recommend_movie("The Avengers")

32335          Drums Across the River
26558         Avengers: Age of Ultron
32951                      Demolition
12406            It's a Free World...
13519              Echelon Conspiracy
32755                        Nice Guy
25534    Kingsman: The Secret Service
11193                   Crime Busters
6919      A Woman Under the Influence
44530        The Most Beautiful Thing
Name: title, dtype: object

We see that for **The Dark Knight**, our system is able to identify it as a Batman film and subsequently recommend other Batman films as its top recommendations. This is not of much use to most people as it doesn't take into considerations very important features such as cast, crew, director and genre, which determine the rating and the popularity of a movie. Someone who liked **The Dark Knight** probably likes it more because of Nolan and would hate **Batman Forever** and every other substandard movie in the Batman Franchise.

Therefore, we are going to use much more suggestive metadata than **Overview** and **Tagline**.

##### 2. Metadata Based Recommender

In [30]:
credits = pd.read_csv('credits.csv')
keywords = pd.read_csv('keywords.csv')

In [31]:
credits.head()

,cast,crew,id
0,"[{'cast_id': 14, 'character': 'Woody (voice)',...","[{'credit_id': '52fe4284c3a36847f8024f49', 'de...",862
1,"[{'cast_id': 1, 'character': 'Alan Parrish', '...","[{'credit_id': '52fe44bfc3a36847f80a7cd1', 'de...",8844
2,"[{'cast_id': 2, 'character': 'Max Goldman', 'c...","[{'credit_id': '52fe466a9251416c75077a89', 'de...",15602
3,"[{'cast_id': 1, 'character': ""Savannah 'Vannah...","[{'credit_id': '52fe44779251416c91011acb', 'de...",31357
4,"[{'cast_id': 1, 'character': 'George Banks', '...","[{'credit_id': '52fe44959251416c75039ed7', 'de...",11862


In [32]:
keywords.head()

,id,keywords
0,862,"[{'id': 931, 'name': 'jealousy'}, {'id': 4290,..."
1,8844,"[{'id': 10090, 'name': 'board game'}, {'id': 1..."
2,15602,"[{'id': 1495, 'name': 'fishing'}, {'id': 12392..."
3,31357,"[{'id': 818, 'name': 'based on novel'}, {'id':..."
4,11862,"[{'id': 1009, 'name': 'baby'}, {'id': 1599, 'n..."


In [33]:
md['id'] = pd.to_numeric(md['id'], errors='coerce').astype('Int64')

In [34]:
md = md.join(credits, on='id', rsuffix='_cr')
md = md.join(keywords, on='id', rsuffix='_kw')

In [35]:
md.columns

Index(['adult', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id',
       'imdb_id', 'original_language', 'original_title', 'overview',
       'popularity', 'poster_path', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'video',
       'vote_average', 'vote_count', 'year', 'tag_and_description', 'cast',
       'crew', 'id_cr', 'id_kw', 'keywords'],
      dtype='object')

In [36]:
md['cast'] = md['cast'].fillna('[]').apply(literal_eval)
md['crew'] = md['crew'].fillna('[]').apply(literal_eval)
md['keywords'] = md['keywords'].fillna('[]').apply(literal_eval)

In [37]:
def get_director(x):
    for i in x:
        if i['job'] == 'Director':
            return i['name']
    return np.nan

In [38]:
md['director'] = md['crew'].apply(get_director)

In [39]:
md['cast'] = md['cast'].apply(lambda x: [i['name'] for i in x] if isinstance(x, list) else [])
md['cast'] = md['cast'].apply(lambda x: x[:3] if len(x) >=3 else x)

In [40]:
md['keywords'] = md['keywords'].apply(lambda x: [i['name'] for i in x] if isinstance(x, list) else [])

In [41]:
def filter_keywords(x):
    s = md.apply(lambda x: pd.Series(x['keywords']),axis=1).stack().reset_index(level=1, drop=True)
    s.name = 'keyword'
    s = s.value_counts()
    s = s[s > 1]
    words = []
    for i in x:
        if i in s:
            words.append(i)
    return words

My approach to building the recommender is going to be extremely *hacky*. What I plan on doing is creating a metadata dump for every movie which consists of **genres, director, main actors and keywords.** I then use a **Count Vectorizer** to create our count matrix as we did in the Description Recommender. The remaining steps are similar to what we did earlier: we calculate the cosine similarities and return movies that are most similar.

These are steps I follow in the preparation of my genres and credits data:
1. **Strip Spaces and Convert to Lowercase** from all our features. This way, our engine will not confuse between **Johnny Depp** and **Johnny Galecki.** 
2. **Mention Director 3 times** to give it more weight relative to the entire cast.

In [42]:
md['cast'] = md['cast'].apply(lambda x: [str.lower(i.replace(" ", "")) for i in x])

In [43]:
md['director'] = md['director'].astype('str').apply(lambda x: str.lower(x.replace(" ", "")))
md['director'] = md['director'].apply(lambda x: [x,x, x])

In [44]:
# stemmer = SnowballStemmer('english')

# md['keywords'] = md['keywords'].apply(filter_keywords)
# md['keywords'] = md['keywords'].apply(lambda x: [stemmer.stem(i) for i in x])
# md['keywords'] = md['keywords'].apply(lambda x: [str.lower(i.replace(" ", "")) for i in x])

In [46]:
md['soup'] = md['cast'] + md['director'] + md['genres'] # + md['keywords']
md['soup'] = md['soup'].apply(lambda x: ' '.join(x))

In [50]:
count = CountVectorizer(analyzer='word', min_df=0.0, stop_words='english')
count_matrix = count.fit_transform(md['soup'])

In [51]:
cosine_sim_arr = cosine_similarity(count_matrix, count_matrix)

We will reuse the get_recommendations function that we had written earlier. Since our cosine similarity scores have changed, we expect it to give us different (and probably better) results. Let us check for **The Dark Knight** again and see what recommendations I get this time around.

In [52]:
recommend_movie('The Dark Knight', cosine_sim_arr=cosine_sim_arr)

43303         The Last Joint Venture
11837    I'm a Cyborg, But That's OK
10550                    The Descent
8743            French Connection II
18252          The Dark Knight Rises
22981                 Need for Speed
20206                 Gangster Squad
17667                     Colombiana
22761                       Sabotage
19024                 Get the Gringo
Name: title, dtype: object

## Collaborative Filtering

Our content based engine suffers from some severe limitations. It is only capable of suggesting movies which are *close* to a certain movie. That is, it is not capable of capturing tastes and providing recommendations across genres.

Also, the engine that we built is not really personal in that it doesn't capture the personal tastes and biases of a user. Anyone querying our engine for recommendations based on a movie will receive the same recommendations for that movie, regardless of who s/he is.

Therefore, in this section, we will use a technique called **Collaborative Filtering** to make recommendations to Movie Watchers. Collaborative Filtering is based on the idea that users similar to a me can be used to predict how much I will like a particular product or service those users have used/experienced but I have not.

I will not be implementing Collaborative Filtering from scratch. Instead, I will use the **Surprise** library that used extremely powerful algorithms like **Singular Value Decomposition (SVD)** to minimise RMSE (Root Mean Square Error) and give great recommendations.

In [237]:
ratings = pd.read_csv('ratings_small.csv')

In [243]:
ratings

,userId,movieId,rating,timestamp
0,1,31,2.5,1260759144
1,1,1029,3.0,1260759179
2,1,1061,3.0,1260759182
3,1,1129,2.0,1260759185
4,1,1172,4.0,1260759205
...,...,...,...,...
99999,671,6268,2.5,1065579370
100000,671,6269,4.0,1065149201
100001,671,6365,4.0,1070940363
100002,671,6385,2.5,1070979663


#### User-based to Item-based

In [244]:
ratings_pivot = ratings.pivot(index='userId',
                              columns='movieId',
                              values='rating')
ratings_pivot

movieId,1,2,3,4,5,6,7,8,9,10,...,161084,161155,161594,161830,161918,161944,162376,162542,162672,163949
userId,,,,,,,,,,,,,,,,,,,,,
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
667,NaN,NaN,NaN,NaN,NaN,4.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
668,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
669,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [246]:
# Filling the missing values
avg_ratings = ratings_pivot.mean(axis=1)
ratings_pivot = ratings_pivot.sub(avg_ratings, axis=0).fillna(0)

In [249]:
cosine_sim_arr = cosine_similarity(ratings_pivot)

In [262]:
cosine_similarity_df = pd.DataFrame(ratings_pivot,
                                    index=ratings_pivot.columns,
                                    columns=ratings_pivot.columns)
cosine_similarity_df.head()

movieId,1,2,3,4,5,6,7,8,9,10,...,161084,161155,161594,161830,161918,161944,162376,162542,162672,163949
movieId,,,,,,,,,,,,,,,,,,,,,
1,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.513158,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,-0.348039,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.09,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [263]:
md.loc[md['title']=='The Godfather']

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,vote_count,year,tag_and_description,cast,crew,id_cr,id_kw,keywords,director,soup
834,False,"{'id': 230, 'name': 'The Godfather Collection'...",6000000,"[Drama, Crime]",http://www.thegodfather.com/,238,tt0068646,en,The Godfather,"Spanning the years 1945 to 1955, a chronicle o...",...,6024.0,1972.0,"Spanning the years 1945 to 1955, a chronicle o...","[stefanodionisi, enricoloverso, elsazylberstein]","[{'credit_id': '52fe43d89251416c750205c5', 'de...",10954.0,10954.0,"[{'id': 380, 'name': 'brother brother relation...","[gérardcorbiau, gérardcorbiau, gérardcorbiau]",stefanodionisi enricoloverso elsazylberstein g...


In [ ]:
cos_sim_series = cosine_similarity_df.loc[834]
ordered_similarities = cos_sim_series.sort_values(ascending=False)
ordered_similarities

In [268]:
recommend_movie("The Godfather")

1178               The Godfather: Part II
44030    The Godfather Trilogy: 1972-1990
1914              The Godfather: Part III
23126                          Blood Ties
31973                    Honor Thy Father
11297                    Household Saints
29567                          Blood Feud
30197                    See You Tomorrow
30358             Quelle strane occasioni
31537                         Sorelle Mai
Name: title, dtype: object